# 06A1 PHA in water (GAFF2, GROMACS)

PHA: GAFF2 from 05A. Water: CHARMM-style TIP3P. Ions: SOD/CLA. Engine: GROMACS.

This is the method used for the polymer-only benchmark (AM1-BCC charges for the existing systems; ABCG2 is the default for new systems). Note: notebook 06A2 (PHA + enzyme) uses GAFF2 + ff19SB in OPC water with OpenMM, and 06B2 uses CGenFF + CHARMM36m with GROMACS, so the PHA force field differs between 06A1 and 06B2.

Every step that runs GROMACS or writes files is behind a `RUN_…`/`WRITE_…` flag that defaults to `False`.

## §1 Inputs (05A prmtop/inpcrd)

The GAFF2 AMBER files from `05A_amber_gaff2_parameters.ipynb` (`gaff2/<SYSTEM>.prmtop`, `gaff2/<SYSTEM>.inpcrd`) and the GROMACS folders `gromacs/dry_polymer/` and `gromacs/solvated_polymer/` next to them.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
from iphasimulator.naming import oligomer_name

system_name = oligomer_name("3HB", 4)
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gaff2_dir = md_root / "gaff2"
gromacs_dir = md_root / "gromacs"
dry_polymer_dir = gromacs_dir / "dry_polymer"
solvated_polymer_dir = gromacs_dir / "solvated_polymer"
prmtop_path = gaff2_dir / f"{system_name}.prmtop"
inpcrd_path = gaff2_dir / f"{system_name}.inpcrd"

{
    "prmtop": prmtop_path,
    "prmtop_exists": prmtop_path.exists(),
    "inpcrd": inpcrd_path,
    "inpcrd_exists": inpcrd_path.exists(),
    "dry_polymer_dir": dry_polymer_dir,
    "solvated_polymer_dir": solvated_polymer_dir,
}

## §2 Convert to GROMACS

`prepare_gromacs_run_folder` converts the AMBER files with ParmEd into a self-contained `gromacs/dry_polymer/` folder (`topol.top` with `[ defaults ] 1 2 yes 0.5 0.83333` and GAFF2 atom types, `step5_input.gro`, `index.ndx`, the polymer mdp files). The next cells check the topology includes and the minimisation inputs.

In [ ]:
from iphasimulator.simulation_gromacs_runner import prepare_gromacs_run_folder

RUN_GROMACS_CONVERSION = False

if RUN_GROMACS_CONVERSION:
    prepared = prepare_gromacs_run_folder(
        prmtop_path,
        inpcrd_path,
        gromacs_dir,
        system_name,
        workflow_type="polymer",
    )
    prepared
else:
    print("Set RUN_GROMACS_CONVERSION = True to regenerate the dry GROMACS workflow folder.")

### Validate Converted Topology

This checks that every `#include` in the dry topology resolves. ParmEd often writes standalone topologies, which is also valid.


In [13]:
from iphasimulator.simulation_gromacs_runner import validate_gromacs_run_folder

if (dry_polymer_dir / "topol.top").exists():
    topology_validation = validate_gromacs_run_folder(dry_polymer_dir)
    print(f"Standalone topology: {topology_validation.is_standalone}")
    print(f"Missing includes: {topology_validation.missing_files}")
    topology_validation
else:
    print(f"Missing dry topology: {dry_polymer_dir / 'topol.top'}")

Standalone topology: True
Missing includes: ()


### Check Dry Minimisation Inputs

The dry minimisation script must be run from inside `gromacs/dry_polymer/`.


In [14]:
from iphasimulator.simulation_gromacs_runner import check_gromacs_minimization_inputs

minimization_check = check_gromacs_minimization_inputs(dry_polymer_dir)
print(f"Ready: {minimization_check.ready}")
print(f"Command: {minimization_check.command_text}")
for path in minimization_check.missing_files:
    print(f"Missing: {path}")

Ready: True
Command: bash run_step6_local.sh


## §3 Add water and ions

`write_gromacs_solvation_files` writes `gromacs/solvated_polymer/` with the packaged water/ion files from `src/iphasimulator/data/gromacs_solvation/` (`TIP3_SOL.itp`, `SOD.itp`, `CLA.itp`, `tip3_ions_atomtypes.itp`, `ions.mdp`) and the scripts `run_solvate_local.sh`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm`. `run_solvate_local.sh` then runs `gmx editconf` (cubic box, 1.2 nm padding), `gmx solvate`, `gmx genion` (neutralise, 0.15 M SOD/CLA) and a `grompp` check of the minimisation input.

In [ ]:
import importlib
import iphasimulator.simulation_gromacs_runner as gromacs_runner

gromacs_runner = importlib.reload(gromacs_runner)
write_gromacs_solvation_files = gromacs_runner.write_gromacs_solvation_files

WRITE_SOLVATION_FILES = False

if WRITE_SOLVATION_FILES:
    solvation_files = write_gromacs_solvation_files(
        gromacs_dir,
        workflow_type="polymer",
        box_padding_nm=1.2,
        ion_concentration_molar=0.15,
        clean=True,
    )
    solvation_files
else:
    print("Set WRITE_SOLVATION_FILES = True to write the solvation scripts into solvated_polymer/ (clean=True resets its generated files).")

### Run Local Solvation When Ready

This step writes water, ions, final coordinates, index files, and `minim_grompp.log`. It is disabled by default. The default solvent group is `SOL`; if your `make_ndx` output uses a different name, run the script with `--solvent-group <name>`.


In [ ]:
import subprocess

RUN_GROMACS_SOLVATION = False

if RUN_GROMACS_SOLVATION:
    result = subprocess.run(
        ["bash", "run_solvate_local.sh"],
        cwd=solvated_polymer_dir,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("Set RUN_GROMACS_SOLVATION = True to run GROMACS solvation and ion insertion.")

## §4 Check topology and coordinates

The final `step5_input.gro` must have as many atoms as `topol.top` describes, and the topology must contain the solvent and ion molecules.

In [9]:
from iphasimulator.simulation_gromacs_runner import (
    validate_gromacs_coordinate_topology_counts,
    validate_gromacs_solvated_topology,
)

if (solvated_polymer_dir / "step5_input.gro").exists():
    count_validation = validate_gromacs_coordinate_topology_counts(solvated_polymer_dir)
    topology_validation = validate_gromacs_solvated_topology(solvated_polymer_dir)
    print(f"Coordinate atoms: {count_validation.coordinate_atom_count}")
    print(f"Expected topology atoms: {count_validation.expected_atom_count}")
    print(f"Counts match: {count_validation.valid}")
    print(topology_validation.molecule_counts)
else:
    print(f"Missing final solvated coordinates: {solvated_polymer_dir / 'step5_input.gro'}")

Coordinate atoms: 7394
Expected topology atoms: 7394
Counts match: True
{'PHA': 1, 'SOL': 2443, 'SOD': 7, 'CLA': 7}


## §5 Optional local minimisation

Runs `bash run_step6_local.sh` in `solvated_polymer/` (`grompp` + `mdrun` for `step6.0_minimization`).

In [ ]:
from iphasimulator.simulation_gromacs_runner import run_gromacs_local_minimization

RUN_LOCAL_MINIMIZATION = False

if RUN_LOCAL_MINIMIZATION:
    result = run_gromacs_local_minimization(solvated_polymer_dir)
    print(result.stdout)
else:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 minimisation locally in solvated_polymer/.")

## §6 Files for 07 (HPC)

### Production Preparation

This notebook prepares local GROMACS minimisation and engine-specific NVT/NPT/production scripts. Submit or benchmark those scripts from notebook 07 only.


### Generated Polymer Run Stages

The generated GROMACS polymer scripts use the following stages after solvation and ion insertion. The conceptual comparison between workflow branches is described at the end of notebook 05.

| Stage | What it does | Simulation length | Why it is included |
| --- | --- | ---: | --- |
| `step6.0_minimization` | Energy minimisation of the solvated/ionised box | Not time-based | Removes close contacts before molecular dynamics starts |
| `step6.1_nvt` | Constant-volume temperature equilibration at 300 K | 100 ps | Assigns/generates velocities and stabilises temperature without changing box volume |
| `step6.2_npt` | Constant-pressure density equilibration at 300 K and 1 bar | 500 ps | Lets the solvent box density and pressure relax before production |
| `step7_production` | Production MD at 300 K and 1 bar | 100 ns by default | Generates the trajectory used by preprocessing and analysis notebooks |

The physical simulation length comes from each `.mdp` file as `nsteps * dt`. The current polymer templates use `dt = 0.002 ps`: `step6.1_nvt.mdp` is 50,000 steps, `step6.2_npt.mdp` is 250,000 steps, and `step7_production.mdp` is 50,000,000 steps.

Wall-clock runtime is hardware-dependent. Use this workflow to iterate quickly and to benchmark polymer-only systems before spending time on more conservative protocols.


In [10]:
{
    "local_minimization_script": solvated_polymer_dir / "run_step6_local.sh",
    "hpc_script_for_07": solvated_polymer_dir / "run_hpc_equilibration_production.slurm",
    "final_start_structure": solvated_polymer_dir / "step5_input.gro",
    "final_topology": solvated_polymer_dir / "topol.top",
}

{'local_minimization_script': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/run_step6_local.sh'),
 'hpc_script_for_07': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/run_hpc_equilibration_production.slurm'),
 'final_start_structure': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/step5_input.gro'),
 'final_topology': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gromacs/solvated_polymer/topol.top')}

## Force fields in the final topology

`[ defaults ]` (`1 2 yes 0.5 0.83333` is the Amber/GAFF2 convention), the GAFF2 atom types of the PHA, and the `[ molecules ]` section of `solvated_polymer/topol.top`.

In [ ]:
def topology_sections(topology_path):
    """[ defaults ], PHA atom types and [ molecules ] from a GROMACS topology and its includes."""
    includes = [topology_path.parent / line.split('"')[1] for line in topology_path.read_text().splitlines()
                if line.strip().startswith("#include")]
    defaults, pha_types, molecules = None, set(), []
    for path in [topology_path] + [p for p in includes if p.exists()]:
        section, molecule = None, None
        for raw in path.read_text().splitlines():
            line = raw.split(";", 1)[0].strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("["):
                section = line.strip("[] ").lower()
                continue
            fields = line.split()
            if section == "defaults" and defaults is None:
                defaults = " ".join(fields)
            elif section == "moleculetype":
                molecule = fields[0]
            elif section == "atoms" and molecule == "PHA" and len(fields) > 1:
                pha_types.add(fields[1])
            elif section == "molecules" and path == topology_path:
                molecules.append((fields[0], int(fields[1])))
    return defaults, sorted(pha_types), molecules

final_topology = solvated_polymer_dir / "topol.top"
if final_topology.exists():
    defaults, pha_types, molecules = topology_sections(final_topology)
    print("[ defaults ]  nbfunc comb-rule gen-pairs fudgeLJ fudgeQQ")
    print("             ", defaults)
    print("PHA atom types (GAFF2):", " ".join(pha_types))
    print("[ molecules ]")
    for name, count in molecules:
        print(f"  {name:<6} {count}")
else:
    print(f"Missing final topology: {final_topology}")